<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu/blob/main/ColabMD_Edu_Colab_Protein_Ligand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu/main/Mascot_ColabMD-Edu.png" height="200" align="right" style="height:240px">

# **ColabMD-Edu: Protein-Ligand Dynamics**

Perform coarse-grained molecular dynamic simulation, convert the trajectories into all-atom, and analyse using GROMACS, PLIP and MM-GBSA. Findings are used to aid in understanding the binding interaction and energy of protein-ligand dynamics. For more details, see bottom of the notebook, checkout the ColabMD-Edu Github and publsihed works  

Che Omar, M. T., Musthafa Kamal, M. F., & Azmi, M. N. (2026). Exploration of benzhydrol analogues of 1'-acetoxychavicol acetate as potential inhibitors of sdrE adhesion protein in *Staphylococcus aureus*: Antimicrobial activity and multi-computational analysis. Computational biology and chemistry, 120(Pt 1), 108618. https://doi.org/10.1016/j.compbiolchem.2025.108618

In [ ]:
# Install dependencies
!apt-get update
!apt-get install -y gromacs
!pip install biopython
!apt-get install python2 -y
!pip install pdbfixer openmm
!pip install py3Dmol
!apt-get install -y openbabel
!pip install rdkit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [14]:
# Create the folders
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Ligand-Preparation/AA"
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Ligand-Preparation/CG"

## Step 1: Ligand Preparation

- All-atom

1. https://cactus.nci.nih.gov/translate/ --> Paste smile --> PDB --> Translate

2. Convert ligand (xxx.pdb) to mol2 file using Open Babel

3. Open CGenFF (https://app.cgenff.com/homepage) > login > upload > download xxx.str > download gromacs file (charmm36.ff)

- Coarse-grained


**All-Atom**

*Open Babel*

In [15]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Ligand-Preparation/AA')

In [ ]:
# Import ligand: xxx.pdb from downloaded folder
from google.colab import files
uploaded = files.upload()

In [ ]:
# Fix .pdb and convert to .mol2 file
!obabel xxx.pdb -O xxx_fixed.pdb -p 7.4 -h
!obabel xxx_fixed.pdb -O xxx.mol2

In [ ]:
# Rename the molecule

*CGenFF Output*

In [ ]:
# Import files: xxx.mol2, xxx.str from downloaded folder
from google.colab import files
uploaded = files.upload()

In [ ]:
# Obtain cgenff_charmm2gmx.py
!wget https://raw.githubusercontent.com/Lemkul-Lab/cgenff_charmm2gmx/main/cgenff_charmm2gmx.py

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-Ligand.git

# 2. Extract the specific .ff folder out to your current working directory
!cp -r ColabMD-Edu_Protein-Ligand/Ligand-preparation/CG/charmm36-feb2026_cgenff-5.0.ff/ .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-Ligand

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d charmm36*

In [ ]:
# Convert into related xxx files using command:

!python cgenff_charmm2gmx.py xxx xxx.mol2 xxx.str charmm36-feb2026_cgenff-5.0.ff

In [ ]:
# Generate xxx.gro file
!gmx editconf -f xxx_ini.pdb -o xxx.gro

**Coarse-grained**

In [18]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Ligand-Preparation/CG')

In [ ]:
# Obtain cg_param_m3_fixed.py and fragments-exp.dat files:
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-Ligand/refs/heads/main/Ligand-preparation/CG/cg_param_m3_fixed.py
!wget https://raw.githubusercontent.com/tasyriqomar/ColabMD-Edu_Protein-Ligand/refs/heads/main/Ligand-preparation/CG/fragments-exp.dat

In [ ]:
# Martini ff paramerization
!python ./cg_param_m3_fixed.py "smiles" xxx.gro xxx.itp 1

In [ ]:
# Generate .pdb file
!gmx editconf -f xxx.gro -o xxx.pdb

## Step 2: Coarse-Grained MD (CGMD)
- Convert all-atom (AA) of protein to coarse-grained (CG) of protein
- Use martinize scripts and insane.py for system preparation


In [22]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-Ligand.git

# 2. Extract the specific CGMD folder out to your current working directory
!cp -r ColabMD-Edu_Protein-Ligand/CGMD .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-Ligand

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d CGMD*

**Protein Preparation**

*Download from [RCSB PDB](https://www.rcsb.org/)*






In [33]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/CGMD')

In [ ]:
# Download the raw PDB file directly from the RCSB servers
!wget https://files.rcsb.org/download/7YIT.pdb

In [ ]:
from Bio.PDB import PDBParser, PPBuilder

# 1. Initialize the parser and load your PDB file
pdb_filename = "7YIT.pdb" # Change to your actual file path
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", pdb_filename)

# 2. Isolate Chain A from the first model
model = structure[0]
if 'A' in model:
    chain_a = model['A']

    # 3. Build the polypeptide sequence
    ppb = PPBuilder()
    sequences = []

    for pp in ppb.build_peptides(chain_a):
        sequences.append(str(pp.get_sequence()))

    # Join sequences (handles cases where there are resolved gaps/loops)
    full_sequence = "".join(sequences)

    print("✨ Chain A Sequence (1-Letter Code):")
    print(full_sequence)
    print(f"\nTotal Residues: {len(full_sequence)}")
else:
    print("❌ Error: Chain A not found in this PDB file.")

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)

In [ ]:
# Import protein: xxx.pdb from downloaded folder
from google.colab import files
uploaded = files.upload()

In [ ]:
# Rename the sequence
!gmx editconf -f OPI.pdb -o fixed.pdb -resnr 55

In [ ]:
# Install depedencies
!pip install vermouth
!sudo apt-get install dssp
!pip install mdtraj

In [ ]:
# Martini3 protein parameterize (Coarse-grained)
!martinize2 -f fixed.pdb -o system.top -x CG.pdb -p backbone -ff martini3001 -elastic -ef 500 -el 0.8 -eu 0.9 -ss C

In [43]:
#Update the topol.top with current martini3 itp files
!sed -i 's/#include "martini.itp"/#include "martini_v3.0.0.itp"\n#include "martini_v3.0.0_ions_v1.itp"\n#include "martini_v3.0.0_solvents_v1.itp"\n#include "fxn.itp"/' system.top

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Ligand-Preparation/CG/xxx.pdb" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/CGMD"

In [47]:
# 1. Strip out END, CONECT, MASTER, and ENDMDL lines from the protein file
!grep -v -E "END|CONECT|MASTER|ENDMDL" CG.pdb > clean.pdb

# 2. Append the ligand file to the bottom
!cat fxn.pdb >> clean.pdb

# 3. Clean up the final file to ensure no trailing CONECT lines from the ligand either (optional)
!grep -v -E "CONECT|MASTER" clean.pdb > CG_fxn.pdb
!rm clean.pdb

In [ ]:
# Generate .gro combine file
!gmx editconf -f CG_fxn.pdb -o CG_fxn.gro

In [ ]:
# Make a box and solvate the system
!python2 insane.py -f CG_fxn.gro -o system.gro -pbc cubic -box 10,10,10 -salt 0.15 -charge auto -sol W

**Coarse-Grained MD Simulation**

In [ ]:
# Grompp minimization.tpr file
!gmx grompp -f minimization.mdp -c system.gro -p system.top -o minimization.tpr -maxwarn 10

In [ ]:
# Run minimization simulation
!gmx mdrun -deffnm minimization -v

In [ ]:
# Make index file
!gmx make_ndx -f minimization.gro -o index.ndx

In [ ]:
# Grompp equlibration.tpr file
!gmx grompp -f equilibration.mdp -c minimization.gro -p system.top -o equilibration.tpr -maxwarn 5 -n index.ndx

In [ ]:
# Run equilibration simulation
!gmx mdrun -deffnm equilibration -v

In [ ]:
# Grompp dynamic.tpr file
!gmx grompp -f dynamic.mdp -c equilibration.gro -p system.top -o dynamic.tpr -maxwarn 5 -n index.ndx

In [ ]:
# Run dynamic simulation
!gmx mdrun -deffnm dynamic -v

**Centered and Fitted Trajectories**

In [ ]:
# Make new index with combine protein and ligand; 1|12
!gmx make_ndx -f dynamic.gro -o index.ndx

In [ ]:
# Center the trajectories
!gmx trjconv -pbc mol -center -ur compact -s dynamic.tpr -n index.ndx -f dynamic.xtc -o dynamic_center.xtc

In [ ]:
# Make new index; BB
!echo -e "a BB\nq" | gmx make_ndx -f equilibration.gro -n index.ndx

In [ ]:
# Fit the trajectories
!gmx trjconv -fit rot+trans -s dynamic.tpr -f dynamic_center.xtc -o dynamic_fitted.xtc -n index.ndx

In [ ]:
# Make multiframe pdb
!gmx trjconv -s dynamic.tpr -n index.ndx -f dynamic_center.xtc -dt 1000 -o dynamic.pdb

## Step 3: Backmapping

- Backmapping CG to AA trajectories

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand')

In [ ]:
# 1. Clone the root repository cleanly
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-Ligand.git

# 2. Extract the specific CGMD folder out to your current working directory
!cp -r ColabMD-Edu_Protein-Ligand/Backmapping .

# 3. Clean up the cloned repository folder to keep your workspace tidy
!rm -rf ColabMD-Edu_Protein-Ligand

# 4. Verify that both the script and the force field folder are in the same directory
!ls -d Backmapping*

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping')

In [ ]:
# Upload files: backward.py, initram.sh, Dr_Tasyriq_V4_M2.sh, Dr_Tasyriq_V4_M2_Redo.sh, count
from google.colab import files
uploaded = files.upload()

**Complex All-Atom Preparation (Gromacs Format)**

In [ ]:
!gmx pdb2gmx -f *.pdb -o aa.gro -ignh

In [ ]:
!chmod +x Dr_Tasyriq_V4_M3.sh initram-v5.sh backward.py

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/Mapping')

In [ ]:
!chmod +x __init__.py

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping')

**Running Backmapping Process**

In [ ]:
!bash Dr_Tasyriq_V4_M3.sh

In [ ]:
!chmod +x Dr_Tasyriq_V4_M3_Redo.sh

In [ ]:
!bash Dr_Tasyriq_V4_M3_Redo.sh

**All-atomic file generation**

- remove unnessary files
- concatenate the trajectories
- generate the AA dynamic.tpr
- generate the AA index.ndx

In [ ]:
# Delete ALL files ending in ns.xtc in a specific folder
!rm "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/"*ns.xtc

rm: cannot remove '/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/*ns.xtc': No such file or directory


In [ ]:
# Delete ALL files ending in ns.gro in a specific folder
!rm "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/"*ns.gro

In [ ]:
# Delete ALL files ending in # in a specific folder
!rm "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/"*#

rm: cannot remove '/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/*#': No such file or directory


In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/all_xtc')

In [ ]:
!gmx trjcat -f *.xtc -o all.xtc

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/gro/aa_88ns.gro" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping"

In [ ]:
# AA .tpr file
!gmx grompp -f dynamic.mdp -c aa_88ns.gro -p topol.top -o dynamic.tpr -maxwarn 5

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/all_xtc"

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/all_xtc')

In [ ]:
# AA .ndx file
!gmx make_ndx -f dynamic.tpr -o index.ndx

## Step 4: Analysis
- Trajectory processing, PyMOL visualization, PLIP interactions, MM-GBSA energy evaluation, PCA-FEL.

In [ ]:
# Create the folder first
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis"

# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/all_xtc/all.xtc" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/all_xtc/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/all_xtc/index.ndx" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis"


In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis')

In [ ]:
!gmx trjconv -s dynamic.tpr -n index.ndx -f all.xtc -dt 1000 -o concatenated.pdb

## Step 4.1: PyMOL

- Installation
- Running
- Video Generation

**PyMOL Installation**

In [ ]:
# Install PyMOL
!apt-get install -y pymol

**PyMOL Execution**

## Step 4.2: PLIP

- Installation
- Running
- Figure Generation

**PLIP Installation**

In [ ]:
# Install conda in Colab 3.12 environment
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# This forces the installation of the libraries into the 3.12 environment
!mamba install -c conda-forge openbabel plip pymol-open-source --no-pin -y

In [ ]:
# This forces the use of the 3.11 environemnt for PLIP and openbabel
!rm /usr/local/conda-meta/pinned
!mamba install -c conda-forge openbabel plip pymol-open-source -y

In [ ]:
# Install of depedencies for the figure generations
!mamba install -c conda-forge matplotlib pandas seaborn openpyxl --yes

**PLIP Execution**

In [ ]:
# Upload files: split_pdb.py
from google.colab import files
uploaded = files.upload()

In [ ]:
# Split the concatenated.pdb
!python split_pdb.py

In [ ]:
# Change directory to split_pdbs
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/split_pdbs')

In [ ]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/split_pdbs/plip.sh" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/split_pdbs/convert_xml_to_json.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/split_pdbs"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/split_pdbs/process_json.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/split_pdbs"


In [ ]:
# Run bash file that execute the PLIP
!bash plip.sh

In [ ]:
# Upload files: convert_xml_to_json.py, process_json.py, percentage_interactions.py, type_interactions_color.py, residue_interactions_color_csv_a.py, timeline_interaction_color.py
from google.colab import files
uploaded = files.upload()

In [ ]:
# Convert *.xml files to a .json file
!python convert_xml_to_json.py

In [ ]:
# Generate folders (CSV) for different type of interaction
!python process_json.py

**Generate PLIP figures**

In [ ]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/split_pdbs')

In [ ]:
# Figure Percentage of Interaction
!python percentage_interactions.py

In [ ]:
# Figure Type of interaction
!python type_interactions_color.py

In [ ]:
# Figure Residue of interaction
!python residue_interactions_color_csv_a.py

In [ ]:
# Figure Timeline of interaction
!python timeline_interaction_color.py

## Step 4.3: MM-GBSA

- Installation
- Running
- Figure Generation

**MM-GBSA Installation**

In [12]:
# Create the folder first
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"

# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/all.xtc" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/index.ndx" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/aa.gro" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/topol.top" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/posre.itp" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/fxn.itp" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/fxn.prm" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"

In [13]:
# Copy the folder: !cp -r /path/to/source_folder /path/to/destination_folder
!cp -r "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/charmm36-feb2026_cgenff-5.0.ff/" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp -r "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Backmapping/Mapping/" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"


In [17]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA')

In [15]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/MMGBSA/mmpbsa.in" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"

In [ ]:
# Upload files: mmpbsa.in from GitHub


In [ ]:
# Install conda in Colab 3.12 environment
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Install AmberTools, MPI, and gmx_MMPBSA from the official channel
!mamba install -c conda-forge ambertools mpi4py gmx_mmpbsa -y

In [ ]:
# This forces the use of the 3.11 environemnt for ambertools and gmx_mmpbsa
!rm /usr/local/conda-meta/pinned
!mamba install -c conda-forge ambertools mpi4py gmx_mmpbsa -y

**MM-GBSA Execution**

In [ ]:
# Run MM-PBSA between Sert protein (1) and fxn (13)
!gmx_MMPBSA -O -i mmpbsa.in -cs dynamic.tpr -ct all.xtc -ci index.ndx -cg 1 13 -cp topol.top -o FINAL_RESULTS_MMPBSA.dat -eo FINAL_RESULTS_MMPBSA.csv -do FINAL_DECOMP_MMPBSA.dat -deo FINAL_DECOMP_MMPBSA.csv

**Generate MM-GBSA table & figures**

In [ ]:
# Delta (Complex-receptor-ligand)
!tail -n 20 FINAL_RESULTS_MMPBSA.dat

In [11]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/MMGBSA/per-residue.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/MMGBSA/heatmap.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/MMGBSA"

In [ ]:
# Generate decomposition bar chart
!python per-residue.py

In [ ]:
# Generate heatmap
!python heatmap.py

## Step 4.4: PCA and FEL

- Installation
- Running
- Figure Generation

**PCA-FEL Installation**

In [20]:
# Create the folder first
!mkdir -p "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/PCA-FEL"

# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/all.xtc" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/PCA-FEL"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/dynamic.tpr" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/PCA-FEL"
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/index.ndx" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/PCA-FEL"


In [21]:
import os
os.chdir('/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/PCA-FEL')

**PCA Execution**

In [ ]:
# Covariance: least square fir (Backbone) and covariance analysis (Backbone)
!gmx covar -f all.xtc -s dynamic.tpr -n index.ndx -o eigenval.xvg -v eigenvec.trr -av average.pdb -l covar.log -b 0 -tu ns

In [ ]:
# Eigenvalue: least square fit (Backbone) and component eigenvector (Backbone)
!gmx anaeig -v eigenvec.trr -f all.xtc -s dynamic.tpr -n index.ndx -comp eigcomp1-2.xvg -rmsf eigrmsf1-2.xvg -2d 2dproj1-2.xvg -b 0 -tu ns -first 1 -last 2
!gmx anaeig -v eigenvec.trr -f all.xtc -s dynamic.tpr -n index.ndx -comp eigcomp1-3.xvg -rmsf eigrmsf1-3.xvg -2d 2dproj1-3.xvg -b 0 -tu ns -first 1 -last 3
!gmx anaeig -v eigenvec.trr -f all.xtc -s dynamic.tpr -n index.ndx -comp eigcomp2-3.xvg -rmsf eigrmsf2-3.xvg -2d 2dproj2-3.xvg -b 0 -tu ns -first 2 -last 3

**FEL Execution**

In [ ]:
# Sham:
!gmx sham -f 2dproj1-2.xvg -notime -ls gibb1-2.xpm

In [25]:
# Copy the file: !cp [source] [destination]
!cp "/content/drive/MyDrive/ColabMD-Edu_Protein-Protein/Analysis/PCA-FEL/xpm2txt.py" "/content/drive/MyDrive/ColabMD-Edu_Protein-Ligand/Analysis/PCA-FEL"

In [26]:
!python2 xpm2txt.py -f gibb1-2.xpm -o FEL.dat

**Generate PCA & FEL figures**

In [ ]:
# Generate PCA plot
!python PCA.py

In [ ]:
# Generate FEL plot
!python FEL.py